# FLUKE Coreference Resolution with DeepSeek R1

This notebook evaluates coreference resolution robustness using DeepSeek R1 open-source reasoning model via OpenRouter API with FLUKE linguistic modifications.

In [2]:
# Standard imports
from datasets import load_dataset
import dspy
import os
import pandas as pd
import json
import glob
import time
import random
from dotenv import load_dotenv
from dspy.evaluate import Evaluate

# Import unified FLUKE utilities
from fluke_reasoning_utils import (
    REASONING_MODELS, REASONING_CONFIGS,
    remove_space, extract_classification_prediction,
    aggregate_results, highlight_drops_and_significance,
    compare_models
)

/Users/hungthinh/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load environment variables
load_dotenv()

# For OpenRouter, we need the OpenRouter API key
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
if not openrouter_api_key:
    print("Warning: OPENROUTER_API_KEY not found in environment variables")
    print("Please set your OpenRouter API key in the .env file")

## DeepSeek R1 Configuration

In [4]:
# Select DeepSeek configuration
CONFIG_NAME = 'deepseek'  # Options: 'deepseek', 'deepseek-lite'
config = REASONING_CONFIGS[CONFIG_NAME]

MODEL_NAME = config['model']
MODEL_ID = REASONING_MODELS[MODEL_NAME]

print(f"Configuration: {CONFIG_NAME}")
print(f"Model: {MODEL_NAME} ({MODEL_ID})")
print(f"Description: {config['description']}")

# Configure DSPy with DeepSeek R1 via OpenRouter
lm = dspy.LM(
    model=MODEL_ID,
    api_key=openrouter_api_key,
    api_base="https://openrouter.ai/api/v1",
    max_tokens=20_000,
    temperature=1  # DeepSeek R1 supports temperature control
)
dspy.configure(lm=lm)

Configuration: deepseek
Model: deepseek-r1 (openrouter/deepseek/deepseek-r1)
Description: Open-source reasoning with DeepSeek R1


## Load Coreference Data

In [5]:
# Load coreference dataset
ds = pd.read_json('../../../data/train_dev_test_data/coref/test.json')
ds = ds.to_dict('records')

print(f"Loaded {len(ds)} coreference samples")

# Split by label
positive_samples = []
negative_samples = []
for i, x in enumerate(ds):
    if x["label"] == 1:
        positive_samples.append((i, x))
    else:
        negative_samples.append((i, x))

print(f"Positive samples (coreferent): {len(positive_samples)}")
print(f"Negative samples (not coreferent): {len(negative_samples)}")

# Combine and shuffle
samples = positive_samples + negative_samples
random.shuffle(samples)

Loaded 1517 coreference samples
Positive samples (coreferent): 500
Negative samples (not coreferent): 1017


In [6]:
# Create examples
examples = [
    dspy.Example({
        "text": remove_space(r["text"]),
        "pronoun": r["pronoun"],
        "candidates": '0: ' + r["candidates"][0] + ', 1: ' + r["candidates"][1],
        "label": r['label']
    }).with_inputs("text", "pronoun", "candidates")
    for i, r in samples
]

# Test example
example = examples[0]
print(f"\nExample text: {example.text}")
print(f"Pronoun: {example.pronoun}")
print(f"Candidates: {example.candidates}")
print(f"Label: {example.label}")


Example text: Will Alvin allow Evin to join, or will he let his male ego get the better of him?
Pronoun: he
Candidates: 0: Alvin, 1: Evin
Label: 0


## Define Task with DeepSeek R1

In [18]:
class DeepSeekCoref(dspy.Signature):
    """Determine the candidate that the given pronoun refers to in the text. Think step by step about grammatical agreement, semantic plausibility, and context. Answer with the index of the candidate."""
    text = dspy.InputField()
    pronoun = dspy.InputField()
    candidates = dspy.InputField()
    label = dspy.OutputField(prefix='Answer:')

class DeepSeekCorefModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.Predict(DeepSeekCoref)

    def forward(self, text, pronoun, candidates):
        max_retries = 3
        for _ in range(max_retries):
            pred = self.prog(text=text, pronoun=pronoun, candidates=candidates)
            parsed_answer = extract_classification_prediction(pred.label)
            if parsed_answer is not None and parsed_answer in ['0', '1']:
                return pred
        # If all retries fail, return last prediction
        return pred

# Initialize module
deepseek_coref = DeepSeekCorefModule()

# Evaluation metric
def eval_metric(true, prediction, trace=None):
    pred = prediction.label
    parsed_answer = extract_classification_prediction(pred)
    return parsed_answer == str(true.label)

In [8]:
# Test single example
pred = deepseek_coref(text=example.text, pronoun=example.pronoun, candidates=example.candidates)
print(f"Text: {example.text}")
print(f"Pronoun: {example.pronoun}")
print(f"Candidates: {example.candidates}")
print(f"True Label: {example.label}")
print(f"Prediction: {pred.label}")
print(f"Correct: {eval_metric(example, pred)}")

Text: Will Alvin allow Evin to join, or will he let his male ego get the better of him?
Pronoun: he
Candidates: 0: Alvin, 1: Evin
True Label: 0
Prediction: 0
Correct: True


## Evaluate Original Dataset

In [10]:
# Test size for DeepSeek R1
TEST_SIZE = 200  # Adjust based on API limits and budget
test_examples = examples

print(f"Evaluating {len(test_examples)} examples with DeepSeek R1...")

evaluate = Evaluate(
    devset=test_examples,
    metric=eval_metric,
    num_threads=2,  # Moderate threading for OpenRouter
    display_progress=True,
    display_table=10,
    return_all_scores=True
)

results = evaluate(deepseek_coref)

# Save results
items = []
for sample in results['results']:
    items.append({
        'text': sample[0]['text'],
        'pronoun': sample[0]['pronoun'],
        'candidates': str(sample[0]['candidates']),
        'label': sample[0]['label'],
        'pred': extract_classification_prediction(sample[1]['label']),
        'raw_output': sample[1]['label']
    })

df_result = pd.DataFrame(items)
output_file = f'../results/coref/{MODEL_NAME}-{CONFIG_NAME}-0shot-coref.csv'
df_result.to_csv(output_file, index=False)

print(f"\nDeepSeek R1 Accuracy: {results['score']:.3f}")
print(f"Results saved to: {output_file}")

Evaluating 1517 examples with DeepSeek R1...
Average Metric: 1236.00 / 1517 (81.5%): 100%|██████████| 1517/1517 [2:22:23<00:00,  5.63s/it] 

2025/08/18 13:39:51 INFO dspy.evaluate.evaluate: Average Metric: 1236 / 1517 (81.5%)


,text,pronoun,candidates,example_label,pred_label,eval_metric
0,"Will Alvin allow Evin to join, or will he let his male ego get the...",he,"0: Alvin, 1: Evin",0,0,✔️ [True]
1,Veronica tries to flirt with Kate but she tells her she is married.,she,"0: Veronica, 1: Kate",1,1,✔️ [True]
2,We took the books to the shops because they were old.,they,"0: the books, 1: the shops",0,0,✔️ [True]
3,"Arnold Schwarzenegger cannot terminate John Conner, because he is ...",he,"0: Arnold Schwarzenegger, 1: John Conner",0,0,✔️ [True]
4,But never for once does Golnar speaks against Durdaana even if she...,she,"0: Golnar, 1: Durdaana",0,0,✔️ [True]
5,Reedburn was in love with Victor although his feelings were not re...,his,"0: Reedburn, 1: Victor",0,0,✔️ [True]
6,Wiriya does not like Nanthagorna because she knows that she likes ...,she,"0: Wiriya, 1: Nanthagorna",0,0,✔️ [True]
7,"Hanna asks Ellen if this is true, and she says no.",she,"0: Hanna, 1: Ellen",1,1,✔️ [True]
8,"Bill Clinton speaks better than George Bush, since he is a practic...",he,"0: Bill Clinton, 1: George Bush",0,0,✔️ [True]
9,"Jessica takes Gloria to sleep, but when she returns, everyone else...",she,"0: Jessica, 1: Gloria",0,0,✔️ [True]



DeepSeek R1 Accuracy: 81.480
Results saved to: ../results/coref/deepseek-r1-deepseek-0shot-coref.csv


## Evaluate Modifications

In [13]:
def evaluate_modified_set(data, program, max_samples=50):
    """Evaluate on modified dataset with DeepSeek R1."""
    limited_data = data[:max_samples] if len(data) > max_samples else data
    
    mod_examples = [
        dspy.Example({
            "text": remove_space(r['modified_text']),
            "original_text": remove_space(r['original_text']),
            "pronoun": r['modified_pronoun'],
            "candidates": '0: ' + r["modified_candidates"][0] + ', 1: ' + r["modified_candidates"][1],
            "label": int(r.get('modified_label') if 'modified_label' in r else r['label']),
            "original_label": int(r['original_label'] if 'original_label' in r else r['label']),
            "type": r['type'] if 'type' in r else None,
            "id": r['index']
        }).with_inputs("text", "pronoun", "candidates")
        for r in limited_data
    ]
    
    evaluate = Evaluate(
        devset=mod_examples,
        metric=eval_metric,
        num_threads=2,  # Moderate threading for OpenRouter
        display_progress=True,
        display_table=1,
        return_all_scores=True
    )
    
    return evaluate(program)

In [ ]:
# Load original predictions
original_pred_file = f'../results/coref/{MODEL_NAME}-{CONFIG_NAME}-0shot-coref.csv'
if os.path.exists(original_pred_file):
    original_pred_ds = pd.read_csv(original_pred_file)
    original_pred_ds['text'] = original_pred_ds['text'].apply(remove_space)
    print(f"Loaded original DeepSeek R1 predictions from {original_pred_file}")
else:
    print("Please run original evaluation first")
    original_pred_ds = None

# Test modifications with DeepSeek R1
json_files = glob.glob('../../../data/modified_data/coref/*_100.json')

print(f"\nTesting {len(json_files)} modifications with DeepSeek R1...")

for json_file in json_files:
    print(f"\nProcessing: {json_file.split('/')[-1]}")
    
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    # Evaluate with sample limit
    results_mod = evaluate_modified_set(data, deepseek_coref, max_samples=150)
    
    # Process results
    items = []
    for sample in results_mod['results']:
        item = {
            'text': sample[0]['text'],
            'original_text': sample[0]['original_text'],
            'pronoun': sample[0]['pronoun'],
            'candidates': str(sample[0]['candidates']),
            'type': sample[0]['type'],
            'modified_label': sample[0]['label'],
            'original_label': sample[0]['original_label'],
            'modified_pred': extract_classification_prediction(sample[1]['label'] if 'label' in sample[1] else '0'),
            # 'raw_output': sample[1]['label'],
            'id': sample[0]['id']
        }
        
        # Find original prediction
        if original_pred_ds is not None:
            matches = original_pred_ds[original_pred_ds.index == item['id']]
            item['original_pred'] = matches.iloc[0]['pred'] if not matches.empty else None
        else:
            item['original_pred'] = None
        
        items.append(item)
    
    df_mod = pd.DataFrame(items)
    mod_name = json_file.split('/')[-1].replace('.json', '')
    output_file = f'../results/coref/{MODEL_NAME}-{CONFIG_NAME}-0shot-{mod_name}.csv'
    df_mod.to_csv(output_file, index=False)
    
    print(f"Accuracy: {results_mod['score']:.3f}")
    print(f"Saved to: {output_file}")
    
    time.sleep(3)  # Rate limiting for OpenRouter

Loaded original DeepSeek R1 predictions from ../results/coref/deepseek-r1-deepseek-0shot-coref.csv

Testing 18 modifications with DeepSeek R1...

Processing: casual_100.json
Average Metric: 7.00 / 11 (63.6%):  11%|█         | 11/100 [02:04<17:38, 11.89s/it]

## Chain-of-Thought with DeepSeek R1

In [ ]:
class CoTDeepSeekCoref(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.ChainOfThought(DeepSeekCoref)

    def forward(self, text, pronoun, candidates):
        return self.prog(text=text, pronoun=pronoun, candidates=candidates)

# Test CoT
cot_deepseek_coref = CoTDeepSeekCoref()
pred_cot = cot_deepseek_coref(text=example.text, pronoun=example.pronoun, candidates=example.candidates)
print("Chain-of-Thought with DeepSeek R1:")
print(f"Text: {example.text}")
print(f"Pronoun: {example.pronoun}")
print(f"Candidates: {example.candidates}")
print(f"\nReasoning: {pred_cot.reasoning if hasattr(pred_cot, 'reasoning') else 'N/A'}")
print(f"\nPrediction: {pred_cot.label}")

## Aggregate Results

In [ ]:
# Aggregate all modification results
result_files = glob.glob(f'results/coref/{MODEL_NAME}-{CONFIG_NAME}-0shot-*_100.csv')

if result_files:
    results_df = aggregate_results(
        result_files,
        task_name='coreference_resolution',
        model_name=f'{MODEL_NAME}-{CONFIG_NAME}'
    )
    
    if not results_df.empty:
        # Display summary
        print(f"\n{MODEL_NAME}-{CONFIG_NAME} Results Summary:")
        print(results_df[['modification', 'original_res', 'modified_res', 'difference', 'samples']])
        
        # Save aggregated results
        output_file = f'results/coref/{MODEL_NAME}-{CONFIG_NAME}-DP.csv'
        results_df.to_csv(output_file, index=False)
        print(f"\nAggregated results saved to: {output_file}")
        
        # Display styled results
        styled_df = results_df.round(3).style.apply(highlight_drops_and_significance, axis=1)
        display(styled_df)
else:
    print("No result files found to aggregate")

## Model Comparison

In [ ]:
# Compare DeepSeek R1 with other models
comparison_files = {
    'DeepSeek-R1': f'results/coref/{MODEL_NAME}-{CONFIG_NAME}-0shot-coref.csv',
    'GPT-5': 'results/coref/gpt-5-standard-0shot-coref.csv',
    'GPT-4o': 'results/coref/gpt4o-0shot-coref.csv',
    'Claude-3.5': 'results/coref/claude-3-5-sonnet-0shot-coref.csv',
    'o3-2025-04-16': 'results/coref/o3-2025-04-16-standard-0shot-coref.csv',
    'Mixtral-8x22B': 'results/coref/mixtral-8x22b-0shot-coref.csv'
}

comparison_df = compare_models(comparison_files, task_name='coreference_resolution')

if not comparison_df.empty:
    print("\nModel Comparison (including DeepSeek R1):")
    print(comparison_df)
    
    # Calculate DeepSeek R1 performance relative to others
    if 'DeepSeek-R1' in comparison_df['Model'].values:
        deepseek_acc = comparison_df[comparison_df['Model'] == 'DeepSeek-R1']['Accuracy'].values[0]
        
        # Compare with closed-source models
        closed_models = ['GPT-5', 'GPT-4o', 'Claude-3.5', 'o3-2025-04-16']
        closed_accs = comparison_df[comparison_df['Model'].isin(closed_models)]['Accuracy'].values
        
        if len(closed_accs) > 0:
            avg_closed = closed_accs.mean()
            gap = deepseek_acc - avg_closed
            print(f"\nDeepSeek R1 Performance: {deepseek_acc:.3f}")
            print(f"Average of closed-source models: {avg_closed:.3f}")
            print(f"Performance gap: {gap:+.3f} ({gap*100:+.1f}%)")
            print(f"\nNote: DeepSeek R1 is an open-source model competing with proprietary systems")
    
    # Highlight best performer
    def highlight_max(s):
        is_max = s == s.max()
        return ['background-color: green; color: white' if v else '' for v in is_max]
    
    styled_comparison = comparison_df.style.apply(highlight_max, subset=['Accuracy'])
    display(styled_comparison)
else:
    print("No comparison data available")

## DeepSeek R1 Performance Analysis

In [ ]:
print(f"\n{'='*60}")
print(f"FLUKE Coreference Resolution with DeepSeek R1 Complete!")
print(f"{'='*60}")

if 'results' in locals():
    print(f"\nBase accuracy: {results[0]:.3f}")

if 'results_df' in locals() and not results_df.empty:
    avg_row = results_df[results_df['modification'] == 'average'].iloc[0]
    print(f"Average robustness drop: {avg_row['difference']:.3f}")
    print(f"Modifications tested: {len(results_df) - 1}")

print(f"\nDeepSeek R1 Configuration: {config['description']}")
print(f"\nKey advantages of DeepSeek R1:")
print("• Open-source model with competitive coreference resolution")
print("• Strong pronoun resolution capabilities")
print("• Good understanding of grammatical agreement")
print("• Cost-effective via OpenRouter API")
print("• Supports temperature control")

print(f"\nFiles saved in: results/coref/")